# wPINNs vs Lax-Friedrichs — Burgers' Equation

Two Riemann problems:
- **Rarefaction**: u₀ = −1 (x≤0), +1 (x>0)
- **Moving shock**: u₀ = 1 (x≤0), 0 (x>0)

Domain: x ∈ [−1, 1], t ∈ [0, 0.45]

Pre-trained wPINNs models live in `RarefactionWave/` (rarefaction) and  
`ShockWave/` (shock — train first with `what_solving = 'Moving'` in `ShockRarEntropy.py`).

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import torch

# Make sure repo modules are importable
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from lax_friedrichs import lax_friedrichs, exact_solution, l1_error, relative_l1_error

## Configuration

In [ ]:
# Paths to best pre-trained wPINNs models.
# Setup_39 / Retrain_0 is the best rarefaction run per best.csv (rel L1 = 0.0175).
# Change SHOCK_MODEL_PATH once you have trained the shock model.
RAREFACTION_MODEL_PATH = "RarefactionWave/Setup_39/Retrain_0/ModelSol.pkl"
SHOCK_MODEL_PATH       = "ShockWave/best/ModelSol.pkl"   # set after training

# Lax-Friedrichs resolution
LF_NX  = 500
LF_CFL = 0.9

# Time slices to compare at
TIME_SLICES = [0.10, 0.25, 0.45]

T_FINAL = 0.45

## Helper: load wPINNs model

In [ ]:
def load_wpinns_model(path):
    """Load a serialised wPINNs solution network. Returns None if not found."""
    if not os.path.exists(path):
        print(f"[wPINNs] Model not found at '{path}' — skipping wPINNs panel.")
        return None
    model = torch.load(path, map_location="cpu", weights_only=False)
    model.eval()
    print(f"[wPINNs] Loaded model from '{path}'")
    return model


def wpinns_predict(model, x_np: np.ndarray, t_val: float) -> np.ndarray:
    """Evaluate model on (t_val, x_np) and return u as a numpy array."""
    t_col = np.full_like(x_np, t_val)
    inp = torch.tensor(np.column_stack([t_col, x_np]), dtype=torch.float32)
    with torch.no_grad():
        out = model(inp).numpy().reshape(-1)
    return out

## Helper: find LF solution at a given time

In [ ]:
def lf_at_time(x_lf, t_vec, U_lf, t_query):
    """Return the LF solution array closest to t_query."""
    idx = int(np.argmin(np.abs(t_vec - t_query)))
    return U_lf[idx], t_vec[idx]

---
## Case 1 — Rarefaction Wave

In [ ]:
case = "Rarefaction"

# --- Lax-Friedrichs ---
x_lf, t_vec_lf, U_lf = lax_friedrichs(case, nx=LF_NX, cfl=LF_CFL)
print(f"LF: {len(t_vec_lf)} time steps, final t = {t_vec_lf[-1]:.4f}")

# --- wPINNs ---
model_rar = load_wpinns_model(RAREFACTION_MODEL_PATH)

In [ ]:
fig, axes = plt.subplots(1, len(TIME_SLICES), figsize=(5 * len(TIME_SLICES), 4), sharey=True)
fig.suptitle("Burgers — Rarefaction Wave", fontsize=13)

for ax, t_q in zip(axes, TIME_SLICES):
    u_ex = exact_solution(case, x_lf, t_q)
    u_lf, t_actual = lf_at_time(x_lf, t_vec_lf, U_lf, t_q)

    ax.plot(x_lf, u_ex,  "k-",  lw=2,   label="Exact")
    ax.plot(x_lf, u_lf,  "r--", lw=1.5, label=f"Lax-Friedrichs")
    if model_rar is not None:
        u_wp = wpinns_predict(model_rar, x_lf, t_q)
        ax.plot(x_lf, u_wp, "b:", lw=1.5, label="wPINNs")

    ax.set_title(f"t = {t_q}")
    ax.set_xlabel("x")
    ax.grid(True, ls=":")
    ax.legend(fontsize=8)

axes[0].set_ylabel("u")
plt.tight_layout()
plt.savefig("comparison_rarefaction_slices.png", dpi=150)
plt.show()

In [ ]:
# Error table at final time T = 0.45
u_ex_final = exact_solution(case, x_lf, T_FINAL)
u_lf_final, _ = lf_at_time(x_lf, t_vec_lf, U_lf, T_FINAL)

print("=== Rarefaction — errors at t = 0.45 ===")
print(f"{'Method':<18} {'L1 error':>12} {'Rel L1 error':>14}")
print("-" * 46)
print(f"{'Lax-Friedrichs':<18} {l1_error(u_lf_final, u_ex_final):>12.6f} {relative_l1_error(u_lf_final, u_ex_final):>14.6f}")
if model_rar is not None:
    u_wp_final = wpinns_predict(model_rar, x_lf, T_FINAL)
    print(f"{'wPINNs':<18} {l1_error(u_wp_final, u_ex_final):>12.6f} {relative_l1_error(u_wp_final, u_ex_final):>14.6f}")

In [ ]:
# Error vs time plot
t_sample = t_vec_lf[::max(1, len(t_vec_lf) // 100)]  # ~100 points
lf_errors = []
wp_errors = []

for t_q in t_sample:
    if t_q == 0:
        continue
    u_ex_t = exact_solution(case, x_lf, t_q)
    u_lf_t, _ = lf_at_time(x_lf, t_vec_lf, U_lf, t_q)
    lf_errors.append(l1_error(u_lf_t, u_ex_t))
    if model_rar is not None:
        u_wp_t = wpinns_predict(model_rar, x_lf, t_q)
        wp_errors.append(l1_error(u_wp_t, u_ex_t))

t_plot = t_sample[1:] if t_sample[0] == 0 else t_sample
plt.figure(figsize=(7, 4))
plt.plot(t_plot[:len(lf_errors)], lf_errors, "r-",  label="Lax-Friedrichs")
if wp_errors:
    plt.plot(t_plot[:len(wp_errors)], wp_errors, "b--", label="wPINNs")
plt.xlabel("t"); plt.ylabel("L1 error")
plt.title("Rarefaction — L1 error vs time")
plt.legend(); plt.grid(True, ls=":")
plt.tight_layout()
plt.savefig("comparison_rarefaction_error_vs_time.png", dpi=150)
plt.show()

---
## Case 2 — Moving Shock

To get a wPINNs model for this case:
1. In `EquationModels/ShockRarEntropy.py` set `self.what_solving = "Moving"`.
2. Run `python wPINNS.py` (or `EnsembleTraining.py`). Best model saves to `ShockWave/`.
3. Update `SHOCK_MODEL_PATH` in the **Configuration** cell above.

In [ ]:
case = "Moving"

# --- Lax-Friedrichs ---
x_lf_s, t_vec_lf_s, U_lf_s = lax_friedrichs(case, nx=LF_NX, cfl=LF_CFL)
print(f"LF: {len(t_vec_lf_s)} time steps, final t = {t_vec_lf_s[-1]:.4f}")

# --- wPINNs ---
model_shock = load_wpinns_model(SHOCK_MODEL_PATH)

In [ ]:
fig, axes = plt.subplots(1, len(TIME_SLICES), figsize=(5 * len(TIME_SLICES), 4), sharey=True)
fig.suptitle("Burgers — Moving Shock", fontsize=13)

for ax, t_q in zip(axes, TIME_SLICES):
    u_ex = exact_solution(case, x_lf_s, t_q)
    u_lf, t_actual = lf_at_time(x_lf_s, t_vec_lf_s, U_lf_s, t_q)

    ax.plot(x_lf_s, u_ex,  "k-",  lw=2,   label="Exact")
    ax.plot(x_lf_s, u_lf,  "r--", lw=1.5, label="Lax-Friedrichs")
    if model_shock is not None:
        u_wp = wpinns_predict(model_shock, x_lf_s, t_q)
        ax.plot(x_lf_s, u_wp, "b:", lw=1.5, label="wPINNs")

    ax.set_title(f"t = {t_q}")
    ax.set_xlabel("x")
    ax.grid(True, ls=":")
    ax.legend(fontsize=8)

axes[0].set_ylabel("u")
plt.tight_layout()
plt.savefig("comparison_shock_slices.png", dpi=150)
plt.show()

In [ ]:
u_ex_final_s = exact_solution(case, x_lf_s, T_FINAL)
u_lf_final_s, _ = lf_at_time(x_lf_s, t_vec_lf_s, U_lf_s, T_FINAL)

print("=== Moving Shock — errors at t = 0.45 ===")
print(f"{'Method':<18} {'L1 error':>12} {'Rel L1 error':>14}")
print("-" * 46)
print(f"{'Lax-Friedrichs':<18} {l1_error(u_lf_final_s, u_ex_final_s):>12.6f} {relative_l1_error(u_lf_final_s, u_ex_final_s):>14.6f}")
if model_shock is not None:
    u_wp_final_s = wpinns_predict(model_shock, x_lf_s, T_FINAL)
    print(f"{'wPINNs':<18} {l1_error(u_wp_final_s, u_ex_final_s):>12.6f} {relative_l1_error(u_wp_final_s, u_ex_final_s):>14.6f}")

---
## LF convergence study (grid refinement)

Shows how LF error decreases as the grid is refined. Useful reference for comparing against wPINNs accuracy.

In [ ]:
nx_vals = [50, 100, 200, 500, 1000]

for case_c in ["Rarefaction", "Moving"]:
    errors = []
    for nx in nx_vals:
        xc, tc, Uc = lax_friedrichs(case_c, nx=nx, cfl=0.9)
        u_lf_c = Uc[-1]
        u_ex_c = exact_solution(case_c, xc, tc[-1])
        errors.append(l1_error(u_lf_c, u_ex_c))

    plt.figure(figsize=(6, 4))
    plt.loglog(nx_vals, errors, "rs-", label="LF L1 error")
    # reference slope -1/2 (expected for LF near shocks)
    ref = errors[0] * (np.array(nx_vals) / nx_vals[0]) ** (-0.5)
    plt.loglog(nx_vals, ref, "k--", alpha=0.5, label="O(nx^{-1/2}) ref")
    plt.xlabel("nx"); plt.ylabel("L1 error")
    plt.title(f"LF convergence — {case_c}")
    plt.legend(); plt.grid(True, ls=":")
    plt.tight_layout()
    plt.savefig(f"lf_convergence_{case_c.lower()}.png", dpi=150)
    plt.show()

    print(f"\n{case_c} LF convergence:")
    print(f"{'nx':>8} {'L1 error':>12}")
    for nx, e in zip(nx_vals, errors):
        print(f"{nx:>8} {e:>12.6f}")

---
## Summary table

In [ ]:
print("=" * 60)
print(f"  SUMMARY — errors at t = {T_FINAL} (L1 / Rel-L1)")
print("=" * 60)

rows = []
for case_s, x_s, t_s, U_s, model_s in [
    ("Rarefaction", x_lf,   t_vec_lf,   U_lf,   model_rar),
    ("Moving shock", x_lf_s, t_vec_lf_s, U_lf_s, model_shock),
]:
    u_ex  = exact_solution(case_s.replace(" shock", "").replace(" ", ""), x_s, T_FINAL)
    u_lf_ = lf_at_time(x_s, t_s, U_s, T_FINAL)[0]
    e_lf  = l1_error(u_lf_, u_ex)
    r_lf  = relative_l1_error(u_lf_, u_ex)
    rows.append((case_s, "Lax-Friedrichs", e_lf, r_lf))
    if model_s is not None:
        u_wp_ = wpinns_predict(model_s, x_s, T_FINAL)
        e_wp  = l1_error(u_wp_, u_ex)
        r_wp  = relative_l1_error(u_wp_, u_ex)
        rows.append((case_s, "wPINNs", e_wp, r_wp))

print(f"{'Case':<16} {'Method':<18} {'L1':>10} {'Rel L1':>10}")
print("-" * 58)
for row in rows:
    print(f"{row[0]:<16} {row[1]:<18} {row[2]:>10.6f} {row[3]:>10.6f}")